# Draining a tank (Torricelli)

*fluids · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/fluids/torricelli_draining)

Water leaves a hole at Torricelli's speed $\sqrt{2gh}$, so the level falls fast at first and slowly at the end. Tanks with different hole sizes are integrated together as one batched tensor and compared with the exact solution $h(t) = (\sqrt{h_0} - \tfrac aA\sqrt{g/2}\,t)^2$.

**Vector form**

$$A\,\frac{dh}{dt} = -a\,\sqrt{2 g h}$$

**Index (tensor) form**

$$\dot h_b = -\frac{a_b}{A}\,\sqrt{2 g\,h_b},\qquad h_b(t) = \Big(\sqrt{h_0} - \frac{a_b}{A}\sqrt{\tfrac g2}\;t\Big)^2,\qquad T_b = \frac{A}{a_b}\sqrt{\frac{2h_0}{g}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $h_b$ | `levels[:, b]` | (T, B) | water level in tank $b$ over time |
| $a_b$ | `circle_area(d)` | (B,) | hole area of tank $b$ — the batch index |
| $A$ | `circle_area(D)` | scalar | cross-section of every tank |
| $\sqrt{2 g h_b}$ | `outflow_speed(h, g)` | (B,) | Torricelli outflow speed |
| $\dot h_b$ | `rhs(h, a, A, g)` | (B,) | right-hand side integrated with RK4 |
| $t\,a_b$ | `einsum('t,b->tb', t, a / A)` | (T, B) | no repeated index ⇒ outer product: every time with every hole |

Torricelli's law follows from Bernoulli between the free surface (at rest, since $A \gg a$) and the jet, both at atmospheric pressure. Real holes squeeze the jet to about 60 % of the hole area (the vena contracta), so real tanks drain more slowly: multiply $a$ by a discharge coefficient $C_d \approx 0.6$. Doubling the hole diameter quarters the drain time; doubling the initial height only multiplies it by $\sqrt2$. The falling jet narrows as it accelerates — continuity again.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/fluids/torricelli_draining/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import analytic_height, drain_time

G = 9.81


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    h0, D = params["h0"], params["tank_diameter"]
    A = tensors.circle_area(D)  # tank cross-section (m²)
    d = tensors.hole_diameters(params["n_tanks"], params["d_min"] / 100, params["d_max"] / 100, dev)  # (B,) in m
    a = tensors.circle_area(d)  # (B,) hole areas
    h_start = tensors.initial_levels(len(d), h0, dev)  # (B,)

    T = drain_time(a, A, h0, G)  # (B,) exact drain times
    times, levels = solve.solve(h_start, a, A, G, t_max=1.05 * T.max().item())
    exact = analytic_height(times, a, A, h0, G)  # (T, B)

    T_num = solve.numerical_drain_time(times, levels)
    for db, Tb, Tn in zip(d.tolist(), T.tolist(), T_num.tolist()):
        print(f"hole {100 * db:4.1f} cm   drain time {Tn:7.1f} s   (exact (A/a) sqrt(2 h0/g) = {Tb:7.1f} s)")
    print(f"largest deviation from the exact h(t): {(levels - exact).abs().max():.2e} m")
    return plot.animate(times, levels, exact, d, D, h0, G, out_dir / "torricelli_draining.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/fluids/torricelli_draining/equation.py
r"""Physics → tensor translation.

Torricelli: water leaves a small hole at the speed of a stone dropped from the surface,
v = \sqrt{2 g h}. The volume lost per second, a v, lowers the level of a tank of cross-section A:

    A \dot h = -a \sqrt{2 g h}

For a batch of tanks b with different hole areas a_b (all elementwise, no summed index):

    \dot h_b = -(a_b / A) \sqrt{2 g h_b}

Separating variables gives the exact solution and the time to empty the tank:

    h_b(t) = (\sqrt{h_0} - (a_b/A) \sqrt{g/2}\; t)^2,      T_b = (A / a_b) \sqrt{2 h_0 / g}
"""

import math

import torch


def outflow_speed(h: torch.Tensor, g: float) -> torch.Tensor:
    """Torricelli's law v_b = sqrt(2 g h_b) (zero once the tank is empty).   (B,) → (B,)"""
    return torch.sqrt(2 * g * h.clamp(min=0.0))


def rhs(h: torch.Tensor, a: torch.Tensor, A: float, g: float) -> torch.Tensor:
    r"""\dot h_b = -(a_b / A) \sqrt{2 g h_b}.   (B,) → (B,)"""
    return -(a / A) * outflow_speed(h, g)


def analytic_height(t: torch.Tensor, a: torch.Tensor, A: float, h0: float, g: float) -> torch.Tensor:
    r"""h_b(t) = (\sqrt{h_0} - (a_b/A)\sqrt{g/2}\,t)^2, clamped at 0 after the drain time.   t: (T,) → (T, B)"""
    root = math.sqrt(h0) - torch.einsum("t,b->tb", t, a / A) * math.sqrt(g / 2)  # outer product t_t a_b
    return root.clamp(min=0.0) ** 2


def drain_time(a: torch.Tensor, A: float, h0: float, g: float) -> torch.Tensor:
    r"""T_b = (A / a_b) \sqrt{2 h_0 / g}.   (B,) → (B,)"""
    return (A / a) * math.sqrt(2 * h0 / g)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/fluids/torricelli_draining/tensors.py
"""Build the concrete tensors: hole diameters and areas, and the initial water levels."""

import math

import torch


def hole_diameters(n: int, d_min: float, d_max: float, device: torch.device) -> torch.Tensor:
    """n hole diameters (m) spread evenly between d_min and d_max: shape (B,)."""
    if n == 1:
        return torch.tensor([d_min], device=device)
    return torch.linspace(d_min, d_max, n, device=device)


def circle_area(d: torch.Tensor | float) -> torch.Tensor | float:
    """Area π d²/4 of a circle of diameter d (works for tensors and plain numbers)."""
    return math.pi * d**2 / 4


def initial_levels(n: int, h0: float, device: torch.device) -> torch.Tensor:
    """Every tank starts full to height h0: shape (B,)."""
    return torch.full((n,), h0, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/fluids/torricelli_draining/solve.py
"""Integrate all tanks together with RK4 until the slowest one is empty."""

import torch

from problems._common.integrators import integrate

from .equation import rhs


def solve(
    h0: torch.Tensor, a: torch.Tensor, A: float, g: float, t_max: float, n_steps: int = 2000
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), levels (T, B)). Levels are clamped at 0 once a tank has emptied."""
    dt = t_max / n_steps
    levels = integrate(lambda h, t: rhs(h, a, A, g), h0, dt, n_steps).clamp(min=0.0)
    times = torch.arange(n_steps + 1, device=h0.device) * dt
    return times, levels


def numerical_drain_time(times: torch.Tensor, levels: torch.Tensor, h_small: float = 1e-6) -> torch.Tensor:
    """First time each level drops below h_small: (B,)."""
    empty = levels < h_small  # (T, B)
    return times[empty.int().argmax(dim=0)]


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/fluids/torricelli_draining/plot.py
"""Left: the tanks draining through holes of different sizes. Right: h(t) against the exact solution."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Polygon, Rectangle

from problems._common.plotting import ACCENT, frame_indices, save_gif

JET_LENGTH = 0.35  # how far below the tank the falling jet is drawn (in units of h0)
WIDEN = 6.0  # holes and jets are drawn this many times wider than in reality


def jet_outline(x0: float, d: float, v0: float, h0: float, g: float) -> np.ndarray:
    """Outline of the falling jet: it narrows as it speeds up (continuity: width² × speed = const)."""
    depth = np.linspace(0, JET_LENGTH * h0, 20)
    speed = np.sqrt(v0**2 + 2 * g * depth)
    half = 0.5 * WIDEN * d * np.sqrt(v0 / speed)
    return np.concatenate([np.column_stack([x0 - half, -depth]), np.column_stack([x0 + half, -depth])[::-1]])


def animate(
    times: torch.Tensor, levels: torch.Tensor, exact: torch.Tensor, d: torch.Tensor, D: float, h0: float, g: float,
    path: Path,
) -> Path:  # fmt: skip
    t = times.cpu().numpy()  # (T,)
    h = levels.cpu().numpy()  # (T, B)
    he = exact.cpu().numpy()  # (T, B)
    diam = d.cpu().numpy()  # (B,)
    n = len(diam)
    gap = 1.5 * D
    colors = [ACCENT[b % len(ACCENT)] for b in range(n)]

    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(8, 3.8), gridspec_kw={"width_ratios": [1.3, 1]})
    ax.set_xlim(-0.2 * D, n * gap)
    ax.set_ylim(-JET_LENGTH * h0 * 1.05, h0 * 1.25)
    ax.set_xlabel("tanks (hole diameter)")
    ax.set_ylabel("h (m)")
    ax.set_xticks([(b + 0.4) * gap for b in range(n)], [f"{100 * x:.1f} cm" for x in diam], fontsize=8)
    for label, color in zip(ax.get_xticklabels(), colors):
        label.set_color(color)  # same colour as the tank's curve on the right
    ax.grid(False)
    water, jets = [], []
    for b in range(n):
        x0 = b * gap
        ax.plot([x0, x0, x0 + D, x0 + D], [h0 * 1.15, 0, 0, h0 * 1.15], color="#374151", lw=2)
        water.append(ax.add_patch(Rectangle((x0, 0), D, h[0, b], fc="#93c5fd", ec="none")))
        jets.append(ax.add_patch(Polygon(np.zeros((2, 2)), fc="#60a5fa", ec="none")))
    clock = ax.set_title(" ")

    ax2.set_xlim(0, t[-1])
    ax2.set_ylim(0, h0 * 1.05)
    ax2.set_xlabel("t (s)")
    ax2.set_ylabel("h (m)")
    for b in range(n):
        ax2.plot(t, he[:, b], color="#374151", lw=0.8, ls="--")
    curves = [ax2.plot([], [], color=colors[b], lw=2)[0] for b in range(n)]
    ax2.set_title("dashed: exact (√h₀ − (a/A)√(g/2) t)²", fontsize=9)

    def update(step: int):
        for b in range(n):
            water[b].set_height(h[step, b])
            x_hole = b * gap + D / 2
            v0 = np.sqrt(2 * g * h[step, b])
            jets[b].set_xy(jet_outline(x_hole, diam[b], v0, h0, g) if v0 > 1e-3 else np.zeros((2, 2)))
            curves[b].set_data(t[: step + 1], h[: step + 1, b])
        clock.set_text(f"t = {t[step]:.0f} s   (holes drawn {WIDEN:.0f}× wider)")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 80), path, fps=16)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_tanks': 4,
    'd_min': 1.0,
    'd_max': 2.5,
    'tank_diameter': 0.4,
    'h0': 1.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.fluids.torricelli_draining")]:
    del sys.modules[name]
main = importlib.import_module("problems.fluids.torricelli_draining.main")
problem_dir = Path("problems/fluids/torricelli_draining")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
